In [11]:
# ==============================
# CHANGE DETECTION STARTER NOTEBOOK
# ==============================
# Classical pipeline: alignment -> normalization -> median background -> diff + SSIM
# Author: ChatGPT (GPT-5)
# ------------------------------

import cv2
import numpy as np
import glob
import matplotlib.pyplot as plt
from skimage.metrics import structural_similarity as ssim
from skimage.exposure import match_histograms
from pathlib import Path

# ------------------------------
# CONFIG
# ------------------------------
IMAGE_DIR = r"C:\Users\Gebruiker\Documents\Falcker\AI\data\Operator Rounds\input\PhotoStream"   # folder with images (same location)
IMAGE_DIR = Path(IMAGE_DIR)
OUTPUT_DIR = r"C:\Users\Gebruiker\Documents\Falcker\AI\data\Operator Rounds\output\ChangeDetection"  # folder to save outputs
N_BACKGROUND = 5                        # how many previous frames to compute median background
ALIGN = True                            # use alignment
DISPLAY_SIZE = 800                      # px width for display
Path(OUTPUT_DIR).mkdir(exist_ok=True)

print( Path(IMAGE_DIR).exists())


True


In [18]:
# ------------------------------
# 1. Load images
# ------------------------------
from email.mime import image


img_files = sorted(list(IMAGE_DIR.glob("**/*.jpg")) + list(IMAGE_DIR.glob("**/*.jpeg")) + list(IMAGE_DIR.glob("**/*.png")))
print(f"Loaded {len(img_files)} images.")

def load_gray(path):
    img = cv2.imread(path)
    if img is None:
        raise FileNotFoundError(path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    return img

images = [load_gray(f) for f in img_files]


Loaded 11 images.


In [19]:
# ------------------------------
# 2. Alignment utilities
# ------------------------------
def align_to_reference(img, ref):
    """Aligns img to ref using ORB feature matching + ECC fine alignment."""
    # --- ORB keypoint matching for initial homography
    orb = cv2.ORB_create(2000)
    kp1, des1 = orb.detectAndCompute(ref, None)
    kp2, des2 = orb.detectAndCompute(img, None)
    if des1 is None or des2 is None:
        return img
    bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
    matches = bf.match(des1, des2)
    matches = sorted(matches, key=lambda x: x.distance)
    if len(matches) < 10:
        return img

    pts1 = np.float32([kp1[m.queryIdx].pt for m in matches]).reshape(-1, 1, 2)
    pts2 = np.float32([kp2[m.trainIdx].pt for m in matches]).reshape(-1, 1, 2)

    H, _ = cv2.findHomography(pts2, pts1, cv2.RANSAC, 5.0)
    if H is not None:
        img_warp = cv2.warpPerspective(img, H, (ref.shape[1], ref.shape[0]))
    else:
        img_warp = img

    # --- ECC fine alignment (optical flow style)
    warp_matrix = np.eye(2, 3, dtype=np.float32)
    try:
        cc, warp_matrix = cv2.findTransformECC(ref, img_warp, warp_matrix,
                                               motionType=cv2.MOTION_EUCLIDEAN)
        img_aligned = cv2.warpAffine(img_warp, warp_matrix,
                                     (ref.shape[1], ref.shape[0]),
                                     flags=cv2.INTER_LINEAR + cv2.WARP_INVERSE_MAP)
    except cv2.error:
        img_aligned = img_warp

    return img_aligned


In [20]:

# ------------------------------
# 3. Photometric normalization
# ------------------------------
def normalize_histogram(img, ref):
    return match_histograms(img, ref)


In [21]:

# ------------------------------
# 4. Difference map
# ------------------------------
def compute_change_map(img, background):
    # SSIM map (1 - ssim)
    score, diff_map = ssim(img, background, full=True)
    diff_map = 1 - diff_map

    # Pixel difference (normalized)
    abs_diff = cv2.absdiff(img, background)
    abs_diff = abs_diff.astype(np.float32) / 255.0

    # Combine maps (tune weights)
    heatmap = 0.6 * abs_diff + 0.4 * diff_map
    heatmap = cv2.GaussianBlur(heatmap, (5,5), 0)
    return heatmap


In [22]:

# ------------------------------
# 5. Threshold & postprocess
# ------------------------------
def postprocess_heatmap(heatmap, thresh=0.25):
    mask = (heatmap > thresh).astype(np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((3,3), np.uint8))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((5,5), np.uint8))
    return mask

In [23]:

# ------------------------------
# 6. Visualization
# ------------------------------
def show_image(title, img):
    plt.figure(figsize=(10,6))
    if img.ndim == 2:
        plt.imshow(img, cmap='gray')
    else:
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis('off')
    plt.show()


In [25]:
def visualize_alignment(ref, aligned):
    diff = cv2.absdiff(ref, aligned)
    diff_color = cv2.applyColorMap(np.clip(diff*4,0,255).astype(np.uint8), cv2.COLORMAP_JET)
    overlay = cv2.addWeighted(cv2.cvtColor(ref, cv2.COLOR_GRAY2BGR), 0.5, diff_color, 0.5, 0)
    plt.imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
    plt.title("Alignment Residuals (red = misaligned)")
    plt.axis("off")
    plt.show()
# Example visualization of alignment residuals

def feature_based_differencing(img1, img2):
    import torch
    import torchvision.models as models
    import torchvision.transforms as T

    # Use pretrained ResNet features
    resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT).eval()
    transform = T.Compose([T.ToTensor(), T.Resize((224,224))])

    feat1 = resnet(transform(img1).unsqueeze(0)).detach()
    feat2 = resnet(transform(img2).unsqueeze(0)).detach()
    diff = torch.norm(feat1 - feat2)
    return diff

In [26]:


# ------------------------------
# 7. Main loop over images
# ------------------------------
background_buffer = []

for i, path in enumerate(img_files):
    curr = images[i]
    ref = images[0]

    if ALIGN and i > 0:
        curr = align_to_reference(curr, ref)

    curr_norm = normalize_histogram(curr, ref)

    if len(background_buffer) >= N_BACKGROUND:
        background = np.median(np.stack(background_buffer[-N_BACKGROUND:]), axis=0).astype(np.uint8)
        heatmap = compute_change_map(curr_norm, background)
        mask = postprocess_heatmap(heatmap, thresh=0.25)

        # overlay mask for visualization
        overlay = cv2.applyColorMap((heatmap*255).astype(np.uint8), cv2.COLORMAP_JET)
        overlay = cv2.addWeighted(cv2.cvtColor(curr, cv2.COLOR_GRAY2BGR), 0.6, overlay, 0.4, 0)

        show_image(f"Change heatmap: {Path(path).name}", overlay)

        # Save results
        cv2.imwrite(f"{OUTPUT_DIR}/{Path(path).stem}_heatmap.png", (heatmap*255).astype(np.uint8))
        cv2.imwrite(f"{OUTPUT_DIR}/{Path(path).stem}_mask.png", (mask*255).astype(np.uint8))

    background_buffer.append(curr_norm)

print("Processing complete.")


    


ValueError: Since image dtype is floating point, you must specify the data_range parameter. Please read the documentation carefully (including the note). It is recommended that you always specify the data_range anyway.